# Notebook 19 — Recursive Memory Compression

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 18 forecast parent-route transitions.

Notebook 19 compresses prototype memory recursively:

- identify redundant prototypes,
- merge similar prototypes,
- preserve route quality,
- compare reconstruction and routing stability before / after compression,
- export compressed prototype bank.

Constraint view:
> adaptive memory should preserve useful distinctions while compressing redundant structure.

## Goals

1. Load Notebook 17/18 hierarchy outputs when available.
2. Load updated prototype bank when available.
3. Score prototype redundancy using feature distance + route similarity.
4. Merge low-distance prototypes into macro-prototypes.
5. Reroute windows through compressed memory.
6. Compare:
   - prototype count,
   - route stability,
   - transition structure,
   - compression ratio,
   - reconstruction proxy quality.
7. Export CSV, JSON, Markdown report, and PNG figures.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load prior outputs

Prefers Notebook 17 hierarchical routing + Notebook 15 updated prototypes.

In [ ]:
route_path = RESULTS_DIR / "notebook17_hierarchical_prototype_routing.csv"
proto_path = RESULTS_DIR / "notebook15_updated_prototypes.csv"
memory_path = RESULTS_DIR / "notebook16_memory_bank_summary.csv"

if route_path.exists():
    routes = pd.read_csv(route_path)
    print("Loaded:", route_path)
else:
    routes = None

if proto_path.exists():
    prototypes = pd.read_csv(proto_path)
    print("Loaded:", proto_path)
else:
    prototypes = None

if memory_path.exists():
    memory = pd.read_csv(memory_path)
    print("Loaded:", memory_path)
else:
    memory = None

if routes is None or prototypes is None:
    print("Missing prior outputs; creating fallback compression dataset.")
    rng = np.random.default_rng(42)
    proto_names = [
        "low_entropy_repeating",
        "sequential_ids",
        "uniform_32bit",
        "zipfian_smallints",
        "clustered_ranges",
        "learned_drift_prototype",
    ]
    prototypes = pd.DataFrame({
        "regime": proto_names,
        "entropy_norm": [0.10, 0.90, 1.00, 0.18, 0.55, 0.51],
        "repetition_ratio": [0.98, 0.00, 0.00, 0.80, 0.98, 0.46],
        "locality_small_delta_ratio": [1.00, 1.00, 0.00, 0.27, 0.00, 0.44],
        "cache_window_reuse_proxy": [0.94, 0.00, 0.00, 0.35, 0.01, 0.41],
        "branch_norm": [0.02, 0.20, 0.72, 0.72, 0.79, 0.64],
        "coherence_score": [0.95, 0.55, 0.08, 0.42, 0.18, 0.45],
        "hardware_pressure_proxy": [0.02, 0.25, 0.88, 0.72, 0.98, 0.69],
    })
    memory = pd.DataFrame({
        "prototype": proto_names,
        "effective_memory_weight": [0.62, 0.48, 0.66, 0.57, 0.50, 0.56],
        "memory_stability_score": [0.56, 0.38, 0.58, 0.49, 0.42, 0.60],
        "usage_count": [32, 22, 36, 26, 20, 64],
    })

    n = 240
    seq = []
    for i in range(n):
        if 90 <= i <= 145:
            proto = "learned_drift_prototype"
        else:
            proto = rng.choice(proto_names[:-1])
        seq.append(proto)
    routes = pd.DataFrame({
        "window_id": np.arange(n),
        "child_route": seq,
        "new_dominant_prototype": seq,
        "updated_policy": [
            {
                "low_entropy_repeating": "coherent_local",
                "sequential_ids": "hybrid",
                "uniform_32bit": "simd",
                "zipfian_smallints": "hybrid",
                "clustered_ranges": "guarded_fallback",
                "learned_drift_prototype": "prototype_recovery",
            }[x] for x in seq
        ],
    })

routes.head(), prototypes.head()

## Normalize schema

In [ ]:
feature_cols = [
    "entropy_norm",
    "repetition_ratio",
    "locality_small_delta_ratio",
    "cache_window_reuse_proxy",
    "branch_norm",
    "coherence_score",
    "hardware_pressure_proxy",
]

protos = prototypes.copy()
if "regime" not in protos.columns:
    protos["regime"] = [f"prototype_{i}" for i in range(len(protos))]

for c in feature_cols:
    if c not in protos.columns:
        protos[c] = 0.5
    protos[c] = pd.to_numeric(protos[c], errors="coerce").fillna(0.5)

if memory is not None:
    mem = memory.copy()
    if "prototype" not in mem.columns and "regime" in mem.columns:
        mem["prototype"] = mem["regime"]
    protos = protos.merge(mem, left_on="regime", right_on="prototype", how="left", suffixes=("", "_mem"))

for c in ["effective_memory_weight", "memory_stability_score", "usage_count"]:
    if c not in protos.columns:
        protos[c] = 1.0
    protos[c] = pd.to_numeric(protos[c], errors="coerce").fillna(0.0)

work = routes.copy().sort_values("window_id").reset_index(drop=True)
if "child_route" not in work.columns:
    work["child_route"] = work.get("new_dominant_prototype", "unknown")
if "updated_policy" not in work.columns:
    work["updated_policy"] = "unknown"

protos[["regime"] + feature_cols + ["effective_memory_weight", "memory_stability_score", "usage_count"]]

## Pairwise redundancy score

Redundancy combines:

- feature similarity,
- policy similarity,
- shared route behavior.

In [ ]:
X = protos[feature_cols].to_numpy(float)
names = list(protos["regime"])
D = pairwise_distances(X)

if np.any(D > 0):
    D_norm = D / D.max()
else:
    D_norm = D

usage = work["child_route"].value_counts(normalize=True).to_dict()
policy_by_proto = work.groupby("child_route")["updated_policy"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else "unknown").to_dict()

pair_rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        feature_similarity = 1.0 - float(D_norm[i, j])
        same_policy = 1.0 if policy_by_proto.get(a) == policy_by_proto.get(b) else 0.0
        usage_balance = 1.0 - abs(usage.get(a, 0.0) - usage.get(b, 0.0))
        redundancy_score = (
            0.65 * feature_similarity +
            0.20 * same_policy +
            0.15 * usage_balance
        )
        pair_rows.append({
            "prototype_a": a,
            "prototype_b": b,
            "feature_distance": float(D[i, j]),
            "feature_similarity": feature_similarity,
            "same_policy": bool(same_policy),
            "usage_balance": usage_balance,
            "redundancy_score": redundancy_score,
        })

pairs = pd.DataFrame(pair_rows)
redundancy_threshold = float(pairs["redundancy_score"].quantile(0.70)) if len(pairs) else 1.0
pairs["merge_recommended"] = pairs["redundancy_score"] >= redundancy_threshold

pairs.sort_values("redundancy_score", ascending=False).head()

## Build compressed macro-prototypes

Uses agglomerative clustering to compress prototype bank.

In [ ]:
original_count = len(protos)
target_count = max(2, int(np.ceil(original_count * 0.67)))
target_count = min(target_count, original_count)

if original_count > 1:
    clusterer = AgglomerativeClustering(n_clusters=target_count, linkage="ward")
    macro_ids = clusterer.fit_predict(X)
else:
    macro_ids = np.array([0])

protos["macro_id"] = macro_ids
protos["macro_prototype"] = ["macro_" + str(i) for i in protos["macro_id"]]

compressed_rows = []
for macro, part in protos.groupby("macro_prototype"):
    weights = part["effective_memory_weight"].to_numpy(float)
    if weights.sum() <= 0:
        weights = np.ones(len(part))
    weights = weights / weights.sum()
    row = {
        "macro_prototype": macro,
        "members": ", ".join(part["regime"].tolist()),
        "member_count": int(len(part)),
        "total_usage_count": float(part["usage_count"].sum()),
        "mean_memory_stability": float(part["memory_stability_score"].mean()),
        "mean_effective_memory_weight": float(part["effective_memory_weight"].mean()),
    }
    for c in feature_cols:
        row[c] = float(np.average(part[c].to_numpy(float), weights=weights))
    compressed_rows.append(row)

compressed = pd.DataFrame(compressed_rows)
compression_ratio = 1.0 - (len(compressed) / max(original_count, 1))

protos[["regime", "macro_prototype"]], compressed

## Reroute windows through compressed memory

In [ ]:
macro_map = protos.set_index("regime")["macro_prototype"].to_dict()

work["macro_route"] = work["child_route"].map(macro_map).fillna("macro_unknown")
work["macro_changed"] = work["macro_route"].ne(work["macro_route"].shift(1)).fillna(False)
work["child_changed"] = work["child_route"].ne(work["child_route"].shift(1)).fillna(False)
work["policy_changed"] = work["updated_policy"].ne(work["updated_policy"].shift(1)).fillna(False)

roll = 15
work["macro_switch_rate"] = work["macro_changed"].rolling(roll, min_periods=1).mean()
work["child_switch_rate"] = work["child_changed"].rolling(roll, min_periods=1).mean()
work["policy_switch_rate"] = work["policy_changed"].rolling(roll, min_periods=1).mean()

work["compressed_route_stability"] = (
    1.0
    - 0.55 * work["macro_switch_rate"]
    - 0.25 * work["child_switch_rate"]
    - 0.20 * work["policy_switch_rate"]
).clip(0, 1)

work[["window_id", "child_route", "macro_route", "updated_policy", "compressed_route_stability"]].head()

## Reconstruction proxy before / after compression

Compressed reconstruction assigns each child prototype to its macro-prototype feature vector.

In [ ]:
proto_features = protos.set_index("regime")[feature_cols]
macro_features = compressed.set_index("macro_prototype")[feature_cols]

def residual_to_macro(proto):
    if proto not in proto_features.index:
        return np.nan
    macro = macro_map.get(proto, None)
    if macro not in macro_features.index:
        return np.nan
    a = proto_features.loc[proto].to_numpy(float)
    b = macro_features.loc[macro].to_numpy(float)
    return float(np.linalg.norm(a - b))

protos["compression_residual"] = protos["regime"].apply(residual_to_macro)
compression_quality = 1.0 / (1.0 + protos["compression_residual"].fillna(0.0))
protos["compression_quality"] = compression_quality

work["macro_compression_residual"] = work["child_route"].map(protos.set_index("regime")["compression_residual"]).fillna(0.0)
work["compression_quality"] = work["child_route"].map(protos.set_index("regime")["compression_quality"]).fillna(1.0)

mean_compression_residual = float(work["macro_compression_residual"].mean())
mean_compression_quality = float(work["compression_quality"].mean())

mean_compression_residual, mean_compression_quality

## Macro transition matrix

In [ ]:
macros = sorted(work["macro_route"].unique())
macro_counts = pd.DataFrame(0, index=macros, columns=macros, dtype=float)

for a, b in zip(work["macro_route"].iloc[:-1], work["macro_route"].iloc[1:]):
    macro_counts.loc[a, b] += 1

macro_transition_probs = macro_counts.div(macro_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
macro_transition_probs

## Export compression tables

In [ ]:
csv_path = RESULTS_DIR / "notebook19_recursive_memory_compression.csv"
json_path = RESULTS_DIR / "notebook19_recursive_memory_compression.json"
proto_csv_path = RESULTS_DIR / "notebook19_prototype_to_macro_map.csv"
compressed_csv_path = RESULTS_DIR / "notebook19_compressed_macro_prototypes.csv"
pairs_csv_path = RESULTS_DIR / "notebook19_redundancy_pairs.csv"
macro_transition_csv_path = RESULTS_DIR / "notebook19_macro_transition_matrix.csv"

work.to_csv(csv_path, index=False)
work.to_json(json_path, orient="records", indent=2)
protos.to_csv(proto_csv_path, index=False)
compressed.to_csv(compressed_csv_path, index=False)
pairs.to_csv(pairs_csv_path, index=False)
macro_transition_probs.to_csv(macro_transition_csv_path)

print("Saved:", csv_path)
print("Saved:", json_path)
print("Saved:", proto_csv_path)
print("Saved:", compressed_csv_path)
print("Saved:", pairs_csv_path)
print("Saved:", macro_transition_csv_path)

## Figure 1 — Prototype to macro projection

In [ ]:
fig_path_1 = FIGURES_DIR / "notebook19_prototype_macro_projection.png"

if len(protos) >= 2:
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X)
else:
    coords = np.zeros((len(protos), 2))

protos["pca_x"] = coords[:, 0]
protos["pca_y"] = coords[:, 1]

plt.figure(figsize=(8, 6))
for macro in sorted(protos["macro_prototype"].unique()):
    part = protos[protos["macro_prototype"] == macro]
    plt.scatter(part["pca_x"], part["pca_y"], s=90, label=macro)
    for _, r in part.iterrows():
        plt.text(r["pca_x"], r["pca_y"], r["regime"], fontsize=8)
plt.xlabel("Prototype PC1")
plt.ylabel("Prototype PC2")
plt.title("Recursive Memory Compression: Prototype → Macro Projection")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_1, dpi=160)
plt.show()

print("Saved:", fig_path_1)

## Figure 2 — Compression ratio summary

In [ ]:
fig_path_2 = FIGURES_DIR / "notebook19_compression_ratio_summary.png"

summary_counts = pd.DataFrame([
    {"category": "original prototypes", "count": original_count},
    {"category": "macro-prototypes", "count": len(compressed)},
])

plt.figure(figsize=(7, 4))
plt.bar(summary_counts["category"], summary_counts["count"])
plt.ylabel("Count")
plt.title(f"Recursive Memory Compression: Compression Ratio = {compression_ratio:.2f}")
plt.tight_layout()
plt.savefig(fig_path_2, dpi=160)
plt.show()

print("Saved:", fig_path_2)

## Figure 3 — Macro route timeline

In [ ]:
fig_path_3 = FIGURES_DIR / "notebook19_macro_route_timeline.png"

labels = sorted(work["macro_route"].unique())
lab_to_id = {lab: i for i, lab in enumerate(labels)}

plt.figure(figsize=(12, 4))
plt.step(work["window_id"], work["macro_route"].map(lab_to_id), where="mid")
plt.yticks(list(lab_to_id.values()), list(lab_to_id.keys()))
plt.xlabel("Window")
plt.ylabel("Macro route")
plt.title("Recursive Memory Compression: Macro Route Timeline")
plt.tight_layout()
plt.savefig(fig_path_3, dpi=160)
plt.show()

print("Saved:", fig_path_3)

## Figure 4 — Child vs macro switch rates

In [ ]:
fig_path_4 = FIGURES_DIR / "notebook19_child_vs_macro_switch_rates.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["child_switch_rate"], label="child switch rate")
plt.plot(work["window_id"], work["macro_switch_rate"], label="macro switch rate")
plt.plot(work["window_id"], work["policy_switch_rate"], label="policy switch rate")
plt.xlabel("Window")
plt.ylabel("Rolling switch rate")
plt.title("Recursive Memory Compression: Child vs Macro Switch Rates")
plt.legend()
plt.tight_layout()
plt.savefig(fig_path_4, dpi=160)
plt.show()

print("Saved:", fig_path_4)

## Figure 5 — Compressed route stability timeline

In [ ]:
fig_path_5 = FIGURES_DIR / "notebook19_compressed_route_stability.png"

plt.figure(figsize=(12, 4))
plt.plot(work["window_id"], work["compressed_route_stability"])
plt.xlabel("Window")
plt.ylabel("Compressed route stability")
plt.title("Recursive Memory Compression: Stability Over Time")
plt.tight_layout()
plt.savefig(fig_path_5, dpi=160)
plt.show()

print("Saved:", fig_path_5)

## Figure 6 — Compression residual by prototype

In [ ]:
fig_path_6 = FIGURES_DIR / "notebook19_compression_residual_by_prototype.png"

plot_df = protos.sort_values("compression_residual")
plt.figure(figsize=(10, 5))
plt.bar(plot_df["regime"], plot_df["compression_residual"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Compression residual")
plt.title("Recursive Memory Compression: Residual by Prototype")
plt.tight_layout()
plt.savefig(fig_path_6, dpi=160)
plt.show()

print("Saved:", fig_path_6)

## Figure 7 — Macro transition matrix

In [ ]:
fig_path_7 = FIGURES_DIR / "notebook19_macro_transition_matrix.png"

plt.figure(figsize=(7, 6))
plt.imshow(macro_transition_probs.values, aspect="auto", vmin=0, vmax=1)
plt.xticks(range(len(macros)), macros, rotation=45, ha="right")
plt.yticks(range(len(macros)), macros)
plt.colorbar(label="Transition probability")
plt.xlabel("Next macro route")
plt.ylabel("Current macro route")
plt.title("Recursive Memory Compression: Macro Transition Matrix")
plt.tight_layout()
plt.savefig(fig_path_7, dpi=160)
plt.show()

print("Saved:", fig_path_7)

## Lab-report summary

In [ ]:
report_path = REPORTS_DIR / "report_19_recursive_memory_compression.md"

summary = {
    "windows": int(len(work)),
    "original_prototype_count": int(original_count),
    "compressed_macro_count": int(len(compressed)),
    "compression_ratio": float(compression_ratio),
    "mean_compression_residual": mean_compression_residual,
    "mean_compression_quality": mean_compression_quality,
    "mean_macro_switch_rate": float(work["macro_switch_rate"].mean()),
    "mean_child_switch_rate": float(work["child_switch_rate"].mean()),
    "mean_compressed_route_stability": float(work["compressed_route_stability"].mean()),
    "merge_recommended_pairs": int(pairs["merge_recommended"].sum()) if len(pairs) else 0,
}

lines = [
    "# Report 19 — Recursive Memory Compression",
    "",
    "This report compresses prototype memory into macro-prototypes while tracking route stability and reconstruction proxy quality.",
    "",
    "Constraint view:",
    "> adaptive memory should preserve useful distinctions while compressing redundant structure.",
    "",
    "## Generated outputs",
    "",
    f"- Window compression CSV: `{csv_path}`",
    f"- Window compression JSON: `{json_path}`",
    f"- Prototype to macro map CSV: `{proto_csv_path}`",
    f"- Compressed macro-prototypes CSV: `{compressed_csv_path}`",
    f"- Redundancy pairs CSV: `{pairs_csv_path}`",
    f"- Macro transition matrix CSV: `{macro_transition_csv_path}`",
    f"- Figure: `{fig_path_1}`",
    f"- Figure: `{fig_path_2}`",
    f"- Figure: `{fig_path_3}`",
    f"- Figure: `{fig_path_4}`",
    f"- Figure: `{fig_path_5}`",
    f"- Figure: `{fig_path_6}`",
    f"- Figure: `{fig_path_7}`",
    "",
    "## Summary",
    "",
    pd.DataFrame([summary]).to_markdown(index=False),
    "",
    "## Compressed macro-prototypes",
    "",
    compressed.to_markdown(index=False),
    "",
    "## Prototype → macro map",
    "",
    protos[["regime", "macro_prototype", "compression_residual", "compression_quality"]].to_markdown(index=False),
    "",
    "## Top redundancy pairs",
    "",
    pairs.sort_values("redundancy_score", ascending=False).head(12).to_markdown(index=False) if len(pairs) else "No redundancy pairs.",
    "",
    "## Interpretation",
    "",
    "- Macro-prototypes reduce memory footprint while preserving coarse routing behavior.",
    "- Compression residual marks prototypes that lose detail under merging.",
    "- Macro switch rate should be lower than child switch rate if compression stabilizes routing.",
    "- Redundancy pairs identify candidates for safe merge or grouped monitoring.",
    "",
    "## Next step",
    "",
    "Notebook 20 can add CGCS-style constraint gating over compressed route memory.",
]

report_path.write_text("\n".join(lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if you are running this notebook in Google Colab and want to download generated outputs.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook19_recursive_memory_compression_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook19_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_19_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))